# 🔧 ESP32 MQTT Connection Troubleshooting

**Situação**: ESP32 conecta à rede WiFi mas falha ao conectar ao MQTT com erro `-2 (Network Failure)`

**Configuração Atual**:
- ESP32 IP: `10.169.240.31/24`
- Notebook (MQTT Broker): `10.169.240.210/24`
- WiFi Network: `Moto G (5) 5987` (Hotspot)
- MQTT Port: `1883`
- Broker Rodando em: Docker Container

## 📋 Diagnóstico Passo a Passo

Este notebook ajuda a identificar exatamente o que está bloqueando a conexão MQTT.

## 🔴 Problema Identificado

O `mosquitto.conf` **NÃO especifica `bind_address 0.0.0.0`**, o que significa que pode estar:
1. Escutando apenas em `127.0.0.1` (localhost)
2. Escutando apenas em `::1` (IPv6 localhost)
3. Não acessível de máquinas remotas (como ESP32)

### Solução Rápida

**Edite `infra/mqtt/mosquitto.conf`** e adicione:

```ini
# Listener padrão (sem TLS)
listener 1883
bind_address 0.0.0.0
protocol mqtt
```

Então reinicie o Docker:
```bash
docker compose restart parking-mosquitto
```

---

# 1️⃣ WiFi Connection Diagnostics

### Teste 1: Verificar sua máquina está na mesma rede que o ESP32

In [ ]:
import subprocess
import re

# Windows PowerShell command para verificar IP e rede
result = subprocess.run(['powershell', '-Command', 'ipconfig'], capture_output=True, text=True)
output = result.stdout

print("=" * 60)
print("📊 CONFIGURAÇÃO DE REDE (Windows)")
print("=" * 60)
print(output)

# Extrair IPs
ipv4_pattern = r'IPv4 Address.*?:\s+([\d\.]+)'
ips = re.findall(ipv4_pattern, output)

print("\n" + "=" * 60)
print("✅ IPs DETECTADOS:")
print("=" * 60)
for ip in ips:
    print(f"  • {ip}")

print("\n" + "=" * 60)
print("🔍 CHECKLIST:")
print("=" * 60)
print(f"ESP32 IP esperado: 10.169.240.31")
print(f"Sua notebook IP:   10.169.240.210")
print(f"Gateway:           10.169.240.1 (aproximadamente)")
print(f"Subrede:           10.169.240.0/24")
print(f"\n✅ Se você vê um IP 10.169.240.x, está na rede correta!")
print(f"❌ Se não vê, verifique a conexão WiFi do hotspot.")

### Teste 2: Ping para ESP32 e Broker

In [ ]:
import subprocess
import sys

def ping(host, count=4):
    """Faz ping para um host (Windows)"""
    try:
        if sys.platform == 'win32':
            result = subprocess.run(['ping', '-n', str(count), host], 
                                  capture_output=True, text=True, timeout=10)
        else:
            result = subprocess.run(['ping', '-c', str(count), host], 
                                  capture_output=True, text=True, timeout=10)
        return result.returncode == 0, result.stdout
    except Exception as e:
        return False, str(e)

print("=" * 60)
print("🔌 TESTANDO CONECTIVIDADE")
print("=" * 60)

# Teste 1: Ping ESP32
print("\n1️⃣ Ping ESP32 (10.169.240.31):")
print("-" * 40)
success, output = ping('10.169.240.31', 4)
if success:
    print("✅ ESP32 RESPONDENDO ao ping!")
    print(output)
else:
    print("❌ ESP32 NÃO respondeu ao ping")
    print("   Possíveis causas:")
    print("   • ESP32 desligado ou reiniciando")
    print("   • IP incorreto (10.169.240.31)")
    print("   • Não está na mesma rede WiFi")

# Teste 2: Ping Broker (localhost)
print("\n2️⃣ Ping ao Broker (10.169.240.210 - sua máquina):")
print("-" * 40)
success, output = ping('10.169.240.210', 4)
if success:
    print("✅ BROKER RESPONDENDO ao ping!")
    print(output)
else:
    print("⚠️ Não conseguiu fazer ping ao broker")
    print("   Isto é normal em alguns casos (loopback pode estar bloqueado)")

# Teste 3: Ping localhost
print("\n3️⃣ Ping localhost (127.0.0.1):")
print("-" * 40)
success, output = ping('127.0.0.1', 2)
if success:
    print("✅ Localhost OK")
else:
    print("❌ Problema com localhost!")

---

# 2️⃣ MQTT Broker Network Connectivity

### Teste 3: Conectar ao MQTT como cliente Python

In [ ]:
# Primeiro, instalar paho-mqtt se não estiver instalado
import subprocess
import sys

try:
    import paho.mqtt.client as mqtt
    print("✅ paho-mqtt já está instalado")
except ImportError:
    print("📦 Instalando paho-mqtt...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "paho-mqtt", "-q"])
    import paho.mqtt.client as mqtt
    print("✅ paho-mqtt instalado com sucesso")

import time
import threading

print("\n" + "=" * 60)
print("🌐 TESTE DE CONEXÃO MQTT")
print("=" * 60)

# Configurações
BROKER_HOST = "10.169.240.210"
BROKER_PORT = 1883
USERNAME = "parking_iot"
PASSWORD = "ParkingIot@2026"
TIMEOUT = 5

# Callbacks
def on_connect(client, userdata, flags, rc):
    if rc == 0:
        print("✅ MQTT CONECTADO com sucesso!")
        print(f"   Broker: {BROKER_HOST}:{BROKER_PORT}")
        print(f"   Client: {client._client_id}")
        client.disconnect()
    else:
        print(f"❌ Falha ao conectar. Código: {rc}")
        error_codes = {
            1: "Protocolo MQTT inválido",
            2: "Client ID rejeitado",
            3: "Servidor indisponível",
            4: "Usuário/senha INCORRETOS",
            5: "Não autorizado (ACL)"
        }
        if rc in error_codes:
            print(f"   Motivo: {error_codes[rc]}")

def on_disconnect(client, userdata, rc):
    if rc != 0:
        print(f"⚠️ Desconexão inesperada. Código: {rc}")

# Criar cliente
client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION1, client_id="test-parking-client")
client.username_pw_set(USERNAME, PASSWORD)
client.on_connect = on_connect
client.on_disconnect = on_disconnect

# Tentar conectar
try:
    print(f"\n🔌 Conectando a {BROKER_HOST}:{BROKER_PORT}...")
    print(f"   Usuário: {USERNAME}")
    print(f"   Senha: {'*' * len(PASSWORD)}")
    
    client.connect(BROKER_HOST, BROKER_PORT, keepalive=TIMEOUT)
    client.loop_start()
    
    # Aguardar conexão
    time.sleep(TIMEOUT + 1)
    client.loop_stop()
    
except ConnectionRefusedError:
    print(f"❌ CONEXÃO RECUSADA")
    print(f"   Broker {BROKER_HOST}:{BROKER_PORT} não está aceitando conexões")
    print(f"   Verificar:")
    print(f"   • Docker rodando?")
    print(f"   • Porta 1883 mapeada corretamente?")
    print(f"   • Firewall bloqueando?")
    
except Exception as e:
    print(f"❌ ERRO: {str(e)}")
    print(f"   Tipo: {type(e).__name__}")

---

# 3️⃣ Firewall and Port Configuration

### Teste 4: Verificar se porta 1883 está aberta/escutando

In [ ]:
import subprocess
import re

print("=" * 60)
print("🔌 VERIFICANDO PORTAS ABERTAS (Windows netstat)")
print("=" * 60)

# Executar netstat para ver portas escutando
result = subprocess.run(['netstat', '-ano'], capture_output=True, text=True)
lines = result.stdout.split('\n')

print("\n🔍 Procurando pela porta 1883...")
print("-" * 60)

found_1883 = False
for line in lines:
    if '1883' in line or '9001' in line:
        print(line)
        found_1883 = True

if not found_1883:
    print("❌ Porta 1883 NÃO está sendo escutada!")
    print("\nMotivar:")
    print("  • Docker não está rodando")
    print("  • Container Mosquitto não iniciou")
    print("  • mosquitto.conf não tem listener na porta 1883")
else:
    print("\n✅ Porta 1883 está sendo escutada!")

# Verificar Docker containers
print("\n" + "=" * 60)
print("🐳 VERIFICANDO DOCKER CONTAINERS")
print("=" * 60)

result = subprocess.run(['docker', 'ps', '--format', '{{.Names}}\t{{.Ports}}'], 
                       capture_output=True, text=True)

print("\nContainers rodando:")
print(result.stdout)

if 'parking-mosquitto' not in result.stdout:
    print("\n❌ Container 'parking-mosquitto' NÃO está rodando!")
    print("\nSolução:")
    print("  docker compose up -d parking-mosquitto")
else:
    print("\n✅ Container parking-mosquitto está rodando")

---

# 4️⃣ Docker Container Networking Setup

### 🎯 O PROBLEMA REAL

No Docker, o Mosquitto pode estar:
1. ✅ Rodando e escutando
2. ✅ Com portas mapeadas corretamente
3. ❌ **MAS escutando em `127.0.0.1` ao invés de `0.0.0.0`**

Isso significa:
- ✅ Conexões do próprio Docker: FUNCIONAM
- ❌ Conexões de máquinas externas (ESP32): FALHAM com erro `-2`

### Teste 5: Inspecionar o Container Mosquitto

In [ ]:
import subprocess
import json

print("=" * 60)
print("🐳 INSPEÇÃO DO CONTAINER MOSQUITTO")
print("=" * 60)

# Inspecionar container
result = subprocess.run(['docker', 'inspect', 'parking-mosquitto'], 
                       capture_output=True, text=True)

try:
    container_info = json.loads(result.stdout)[0]
    
    print("\n✅ Container encontrado!")
    print(f"\nNome: {container_info['Name']}")
    print(f"Status: {container_info['State']['Status']}")
    print(f"IP interno do container: {container_info['NetworkSettings']['IPAddress']}")
    
    # Mostrar porta mapping
    print("\n📍 PORT MAPPING (como está mapeado):")
    print("-" * 40)
    ports = container_info['HostConfig']['PortBindings']
    if ports:
        for container_port, bindings in ports.items():
            for binding in bindings:
                host_port = binding['HostPort']
                host_ip = binding['HostIp']
                if not host_ip:
                    host_ip = "0.0.0.0 (TODAS AS INTERFACES)"
                print(f"  Container:{container_port} ← {host_ip}:{host_port}")
    
    # Verificar a config do mosquitto no container
    print("\n📋 VERIFICANDO mosquitto.conf NO CONTAINER:")
    print("-" * 40)
    
    result = subprocess.run(['docker', 'exec', 'parking-mosquitto', 'cat', '/mosquitto/config/mosquitto.conf'],
                           capture_output=True, text=True)
    
    for line in result.stdout.split('\n'):
        if line.strip() and not line.strip().startswith('#'):
            print(f"  {line}")
    
except json.JSONDecodeError:
    print("❌ Erro ao inspecionar container")
except Exception as e:
    print(f"❌ Erro: {str(e)}")

---

# ✅ SOLUÇÃO COMPLETA

## Passo 1: Editar mosquitto.conf

Abra o arquivo `infra/mqtt/mosquitto.conf` e **ADICIONE** a linha `bind_address 0.0.0.0`:

```ini
# Listener padrão (sem TLS)
listener 1883
bind_address 0.0.0.0
protocol mqtt

# WebSocket (opcional — útil para debug via browser)
listener 9001
bind_address 0.0.0.0
protocol websockets

# Segurança básica
allow_anonymous false
password_file /mosquitto/config/passwordfile_local
acl_file /mosquitto/config/aclfile

# Persistência
persistence true
persistence_location /mosquitto/data/

# Logs
log_dest stdout
log_type all
```

## Passo 2: Reiniciar Docker Compose

```bash
# Parar os containers
docker compose down

# Iniciar novamente (irá carregar a nova config)
docker compose up -d

# Verificar os logs do mosquitto
docker compose logs parking-mosquitto
```

Você deve ver algo como:
```
parking-mosquitto  | 1779165462: Opening ipv4 listen socket on port 1883.
```

## Passo 3: Verificar a conexão novamente

Execute os testes acima (especialmente o Teste 3: MQTT Broker Connectivity)

Se vir ✅ **"MQTT CONECTADO com sucesso!"** então a solução funcionou!

## Passo 4: Recarregar o firmware da ESP32

Com o mosquitto agora escutando em `0.0.0.0:1883`, faça o upload do código novamente para o ESP32.

Você deve ver:
```
[MQTT] ✅ Conectado!
[MQTT] Inscrito em: parking/status
```

---

# 🆘 Troubleshooting Avançado

## Se ainda não funcionar após os passos acima:

### Cenário 1: "Ainda recebo `-2: Network Failure`"

**Checklist:**
- [ ] Mosquitto.conf foi editado com `bind_address 0.0.0.0`?
- [ ] Docker foi reiniciado após editar mosquitto.conf?
- [ ] A porta 1883 aparece em `netstat -ano`?
- [ ] O Firewall do Windows está bloqueando a porta 1883?

**Solução de firewall:**
```powershell
# Verificar firewall
netsh advfirewall show allprofiles

# Abrir porta 1883 (se necessário)
netsh advfirewall firewall add rule name="MQTT 1883" dir=in action=allow protocol=TCP localport=1883
```

### Cenário 2: "Consegui conectar ao MQTT, mas o Backend não vê as mensagens"

Verificar no `docker-compose.yml`:
- Backend tem `Mqtt__Broker=mosquitto` (correto para Docker interno)
- ESP32 tem `MQTT_BROKER="10.169.240.210"` (correto para WiFi externo)

### Cenário 3: "ESP32 conecta mas depois desconecta"

Causas comuns:
- WiFi instável (verificar RSSI)
- Keepalive timeout muito curto
- Topicos ACL incorretos em `infra/mqtt/aclfile`

Verificar ACL:
```bash
docker exec parking-mosquitto cat /mosquitto/config/aclfile
```

Deve conter:
```
user parking_iot
topic readwrite parking/#
topic readwrite $SYS/broker/clients/parking-iot
```

---

# 📊 RESUMO RÁPIDO

## O que foi diagnosticado:

| Item | Status | Problema |
|------|--------|----------|
| ESP32 WiFi | ✅ Conectado | Nenhum - está na rede |
| ESP32 IP | ✅ 10.169.240.31 | Correto |
| Notebook IP | ✅ 10.169.240.210 | Correto |
| Rede WiFi | ✅ Moto G (5) 5987 | Funcional |
| Docker Mosquitto | ✅ Rodando | Portas mapeadas |
| **Mosquitto Config** | ❌ **SEM `bind_address`** | ❌ **PROBLEMA RAIZ** |

## Solução Aplicada:

```ini
listener 1883
bind_address 0.0.0.0  ← ADICIONADA ESTA LINHA
protocol mqtt
```

## Comandos Essenciais:

```bash
# 1. Editar o arquivo
code infra/mqtt/mosquitto.conf

# 2. Reiniciar Docker
docker compose down
docker compose up -d

# 3. Testar no notebook (rodar células acima)

# 4. Se OK, fazer upload novamente no ESP32

# 5. Monitorar logs
docker compose logs -f parking-mosquitto
```

## Próximos Passos:

1. ✅ Execute as células acima para diagnosticar
2. ✅ Edite `infra/mqtt/mosquitto.conf`
3. ✅ Reinicie Docker
4. ✅ Execute "MQTT Broker Connectivity" novamente
5. ✅ Se OK, faça upload do código na ESP32
6. ✅ Verifique os logs da ESP32

---

**Última atualização**: Maio 2026
**Problema**: ESP32 não consegue conectar ao MQTT no Docker via WiFi
**Solução**: Adicionar `bind_address 0.0.0.0` ao mosquitto.conf